<a href="https://colab.research.google.com/github/mr-zero-000/Statistical-Learning-e23034/blob/main/Assignment%205/Part%2003/Part_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Parametric Factor Partition Engine
===================================
End-to-end Factor Analysis (FA) pipeline with Varimax rotation,
Thomson MMSE score projection, and a 2×2 Plotly diagnostic dashboard.

Dependencies:
    pip install numpy pandas scipy scikit-learn plotly
"""

from __future__ import annotations

import logging
import warnings
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.linalg import svd
from sklearn.utils.extmath import randomized_svd

warnings.filterwarnings("ignore", category=UserWarning)
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger(__name__)

# ─────────────────────────────────────────────────────────────────────────────
# 1. RESULT CONTAINER
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class FAResult:
    """
    Immutable container for all computed FA artefacts.

    Attributes
    ----------
    loadings_rotated : ndarray (m, k)
        Varimax-rotated factor loading matrix Λ_rotated.
    communalities : ndarray (m,)
        Per-channel shared variance vector h².
    uniquenesses : ndarray (m,)
        Per-channel noise floor vector φ².
    factor_scores : ndarray (n, k)
        Thomson MMSE latent score projections F.
    factor_score_vars : ndarray (k,)
        Empirical variance of each factor score column (ddof=1).
    sensor_labels : list[str]
        Ordered list of sensor / feature channel names.
    n_obs : int
        Number of observations used.
    n_features : int
        Number of feature channels m.
    n_factors : int
        Number of extracted latent factors k.
    avg_communality_pct : float
        System-wide mean communality percentage.
    avg_uniqueness_pct : float
        System-wide mean uniqueness percentage.
    rotation_matrix : ndarray (k, k)
        Orthogonal Varimax rotation matrix T.
    """
    loadings_rotated:    np.ndarray
    communalities:       np.ndarray
    uniquenesses:        np.ndarray
    factor_scores:       np.ndarray
    factor_score_vars:   np.ndarray
    sensor_labels:       list[str]
    n_obs:               int
    n_features:          int
    n_factors:           int
    avg_communality_pct: float
    avg_uniqueness_pct:  float
    rotation_matrix:     np.ndarray


# ─────────────────────────────────────────────────────────────────────────────
# 2. Z-SCORE MAPPING LAYER
# ─────────────────────────────────────────────────────────────────────────────

def zscore_standardize(
    df: pd.DataFrame,
    zero_var_floor: float = 1e-15,
) -> tuple[np.ndarray, list[str]]:
    """
    Isolate numeric columns and standardize into standard normal space.

    Z ∈ ℝ^{n×m}  using unbiased sample standard deviation (ddof=1).
    Zero-variance channels are capped at `zero_var_floor` to prevent
    divide-by-zero runtime exceptions.

    Parameters
    ----------
    df : pd.DataFrame
        Arbitrary input data frame (may contain non-numeric columns).
    zero_var_floor : float
        Minimum σ boundary threshold (default 10⁻¹⁵).

    Returns
    -------
    Z : ndarray (n, m)
        Standardized observation matrix.
    labels : list[str]
        Names of the isolated numeric feature channels.
    """
    numeric_df = df.select_dtypes(include=[np.number])
    labels = numeric_df.columns.tolist()
    X = numeric_df.values.astype(float)

    means = X.mean(axis=0)
    stds  = X.std(axis=0, ddof=1)

    # Zero-variance guardrail
    zero_mask = stds < zero_var_floor
    if zero_mask.any():
        bad_channels = [labels[i] for i, m in enumerate(zero_mask) if m]
        log.warning(
            "[GUARDRAIL] Zero-variance channels detected: %s — "
            "σ capped to %.0e", bad_channels, zero_var_floor
        )
        stds = np.where(zero_mask, zero_var_floor, stds)

    Z = (X - means) / stds
    log.info("[Z-SCORE]  Z ∈ ℝ^{%d×%d}  |  ddof=1  |  σ_floor=%.0e",
             *Z.shape, zero_var_floor)
    return Z, labels


# ─────────────────────────────────────────────────────────────────────────────
# 3. DIMENSIONALITY CONSTRAINT LAYER
# ─────────────────────────────────────────────────────────────────────────────

def validate_dimensions(n: int, m: int, k: int) -> None:
    """
    Enforce two hard dimensionality constraints before decomposition.

    Rule 1 — Underdetermined system:
        Reject if n ≤ m (fewer observations than channels).
    Rule 2 — Latent space overflow:
        Reject if k ≥ m (factors must be strictly fewer than features).

    Parameters
    ----------
    n : int   Number of observation snapshots.
    m : int   Number of sensor feature channels.
    k : int   Requested latent factor dimension.

    Raises
    ------
    ValueError
        With a descriptive message for each violated constraint.
    """
    if n <= m:
        raise ValueError(
            f"[DIM ERROR] Underdetermined system: observation count n={n} "
            f"must be strictly greater than feature count m={m}. "
            f"Collect at least {m + 1} snapshots before decomposition."
        )
    if k >= m:
        raise ValueError(
            f"[DIM ERROR] Latent space overflow: requested factor count k={k} "
            f"must be strictly less than feature dimension m={m}. "
            f"Reduce k to at most {m - 1}."
        )
    log.info("[DIM-CHECK] n=%d > m=%d ✓  |  k=%d < m=%d ✓", n, m, k, m)


# ─────────────────────────────────────────────────────────────────────────────
# 4. VARIMAX ROTATION
# ─────────────────────────────────────────────────────────────────────────────

def varimax_rotation(
    L: np.ndarray,
    max_iter: int = 1_000,
    tol: float = 1e-8,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Orthogonal Varimax rotation via iterative pairwise Givens sweeps.

    Maximises  Σ_r [ (1/m) Σ_j h_{jr}⁴ − ((1/m) Σ_j h_{jr}²)² ]
    where h_{jr} = λ_{jr} / communality_j (Kaiser normalisation).

    Parameters
    ----------
    L : ndarray (m, k)
        Initial (unrotated) factor loading matrix.
    max_iter : int
        Maximum number of full sweep iterations.
    tol : float
        Convergence tolerance on the rotation increment.

    Returns
    -------
    L_rot : ndarray (m, k)
        Varimax-rotated loading matrix Λ_rotated.
    T : ndarray (k, k)
        Accumulated orthogonal rotation matrix.
    """
    m, k = L.shape
    T     = np.eye(k)
    L_rot = L.copy()

    for iteration in range(max_iter):
        old_T = T.copy()
        for p in range(k - 1):
            for q in range(p + 1, k):
                u = L_rot[:, p] ** 2 - L_rot[:, q] ** 2
                v = 2 * L_rot[:, p] * L_rot[:, q]
                A = np.sum(u)
                B = np.sum(v)
                C = np.sum(u ** 2 - v ** 2)
                D = 2 * np.sum(u * v)
                num = D - 2 * A * B / m
                den = C - (A ** 2 - B ** 2) / m
                theta = 0.25 * np.arctan2(num, den)
                c, s  = np.cos(theta), np.sin(theta)
                # Apply Givens rotation
                G           = np.eye(k)
                G[p, p]     =  c;  G[p, q] = s
                G[q, p]     = -s;  G[q, q] = c
                L_rot       = L_rot @ G
                T           = T     @ G

        delta = np.max(np.abs(T - old_T))
        if delta < tol:
            log.info("[VARIMAX]  Converged at iteration %d (Δ=%.2e)", iteration + 1, delta)
            break
    else:
        log.warning("[VARIMAX]  Max iterations (%d) reached without convergence.", max_iter)

    return L_rot, T


# ─────────────────────────────────────────────────────────────────────────────
# 5. INFORMATION DECOMPOSITION LAYER
# ─────────────────────────────────────────────────────────────────────────────

def fit_factor_analysis(
    Z: np.ndarray,
    k: int,
    n_power_iter: int = 4,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Fit a Factor Analysis model via SVD + Varimax rotation.

    Decomposes the sample correlation matrix R = ZᵀZ / (n-1) using
    randomised SVD to extract the top-k principal loading directions,
    then rotates them with Varimax to achieve simple structure.

    Parameters
    ----------
    Z : ndarray (n, m)
        Standardized observation matrix.
    k : int
        Number of latent factors to extract.
    n_power_iter : int
        Power iterations for the randomised SVD (accuracy vs speed).

    Returns
    -------
    L_rot : ndarray (m, k)
        Varimax-rotated factor loadings Λ_rotated.
    h2 : ndarray (m,)
        Communality vector — total shared variance per channel.
    phi2 : ndarray (m,)
        Uniqueness vector — residual noise floor per channel.
    T : ndarray (k, k)
        Orthogonal rotation matrix.
    """
    n, m = Z.shape

    # Sample correlation matrix
    R = (Z.T @ Z) / (n - 1)
    log.info("[SVD]      Computing top-%d eigenpairs of R ∈ ℝ^{%d×%d}", k, m, m)

    # Randomised SVD (stable for small k relative to m)
    U, S, _ = randomized_svd(R, n_components=k, n_iter=n_power_iter, random_state=42)

    # Initial loading matrix: L = U * diag(sqrt(max(λ-1, ε)))
    # subtract 1 to remove the diagonal noise assumption (PAF-style)
    eigenvalues_adj = np.maximum(S - 1.0, 1e-6)
    L = U * np.sqrt(eigenvalues_adj)[np.newaxis, :]
    log.info("[LOADINGS] Initial loadings L ∈ ℝ^{%d×%d}  |  λ_adj ∈ [%.3f, %.3f]",
             m, k, eigenvalues_adj.min(), eigenvalues_adj.max())

    # Varimax rotation
    L_rot, T = varimax_rotation(L)

    # Clamp loadings to [-1, 1] for numerical stability
    L_rot = np.clip(L_rot, -1.0, 1.0)

    # Communality: h²_j = Σ_r λ²_{jr}   (capped at 0.99 to keep φ² > 0)
    h2   = np.clip(np.sum(L_rot ** 2, axis=1), 0.0, 0.99)
    # Uniqueness: φ²_j = 1 − h²_j
    phi2 = 1.0 - h2

    log.info("[DECOMP]   h² ∈ [%.3f, %.3f]  |  φ² ∈ [%.3f, %.3f]",
             h2.min(), h2.max(), phi2.min(), phi2.max())
    return L_rot, h2, phi2, T


# ─────────────────────────────────────────────────────────────────────────────
# 6. LATENT SCORE PROJECTION (Thomson MMSE)
# ─────────────────────────────────────────────────────────────────────────────

def project_factor_scores(
    Z: np.ndarray,
    L_rot: np.ndarray,
    R: np.ndarray,
) -> np.ndarray:
    """
    Thomson Minimum Mean Squared Error (MMSE) regression score estimator.

    Computes latent factor states via:
        F = Z · R⁻¹ · Λ_rotated

    where R⁻¹ is the inverse of the sample correlation matrix.
    Falls back to the Moore-Penrose pseudoinverse if R is singular.

    Parameters
    ----------
    Z : ndarray (n, m)
        Standardized observation matrix.
    L_rot : ndarray (m, k)
        Varimax-rotated loading matrix.
    R : ndarray (m, m)
        Sample correlation matrix.

    Returns
    -------
    F : ndarray (n, k)
        Reconstructed latent factor score matrix.
    """
    try:
        R_inv = np.linalg.inv(R)
        log.info("[SCORES]   R⁻¹ computed via exact inversion (cond=%.2e)",
                 np.linalg.cond(R))
    except np.linalg.LinAlgError:
        R_inv = np.linalg.pinv(R)
        log.warning("[SCORES]   Singular R detected — using Moore-Penrose pseudoinverse.")

    W = R_inv @ L_rot          # Regression weight matrix  (m, k)
    F = Z @ W                  # Factor score matrix        (n, k)
    log.info("[SCORES]   F ∈ ℝ^{%d×%d}  |  F = Z · R⁻¹ · Λ_rot", *F.shape)
    return F


# ─────────────────────────────────────────────────────────────────────────────
# 7. PERFORMANCE DIAGNOSTIC TELEMETRY
# ─────────────────────────────────────────────────────────────────────────────

def print_diagnostics(
    h2: np.ndarray,
    phi2: np.ndarray,
    factor_vars: np.ndarray,
    sensor_labels: list[str],
) -> tuple[float, float]:
    """
    Emit a structured diagnostic telemetry breakdown to the logging console.

    Computes and prints:
        Average System Communality % = (1/m) Σ h²_j × 100
        Average System Uniqueness % =  (1/m) Σ φ²_j × 100

    Parameters
    ----------
    h2 : ndarray (m,)
        Communality vector.
    phi2 : ndarray (m,)
        Uniqueness vector.
    factor_vars : ndarray (k,)
        Empirical variance of each factor score column (ddof=1).
    sensor_labels : list[str]
        Channel names for per-sensor breakdown.

    Returns
    -------
    avg_comm_pct : float
    avg_uniq_pct : float
    """
    m = len(h2)
    avg_comm_pct = float(np.mean(h2) * 100)
    avg_uniq_pct = float(np.mean(phi2) * 100)

    sep = "═" * 52
    log.info("\n╔%s╗", sep)
    log.info("║  FACTOR ANALYSIS — SYSTEM DIAGNOSTIC TELEMETRY     ║")
    log.info("╠%s╣", sep)
    log.info("║  Average System Communality %%  : %6.2f %%            ║", avg_comm_pct)
    log.info("║  Average System Uniqueness  %%  : %6.2f %%            ║", avg_uniq_pct)
    log.info("╠%s╣", sep)
    log.info("║  %-14s  %8s  %8s  %8s  ║", "Channel", "h²", "φ²", "h²%")
    log.info("║  %-14s  %8s  %8s  %8s  ║", "─" * 14, "─" * 8, "─" * 8, "─" * 8)
    for label, h, p in zip(sensor_labels, h2, phi2):
        log.info("║  %-14s  %8.4f  %8.4f  %7.1f%%  ║", label[:14], h, p, h * 100)
    log.info("╠%s╣", sep)
    log.info("║  Factor Score Variances (ddof=1):                   ║")
    for i, v in enumerate(factor_vars):
        log.info("║    Factor %-2d  :  %.4f                               ║", i + 1, v)
    log.info("╚%s╝\n", sep)
    return avg_comm_pct, avg_uniq_pct


# ─────────────────────────────────────────────────────────────────────────────
# 8. MAIN FA ENGINE ENTRYPOINT
# ─────────────────────────────────────────────────────────────────────────────

def run_factor_analysis(
    df: pd.DataFrame,
    k: int,
    zero_var_floor: float = 1e-15,
) -> FAResult:
    """
    Full end-to-end Factor Analysis pipeline.

    Orchestrates all layers in order:
        Z-score mapping → Dimensionality validation → SVD decomposition →
        Varimax rotation → Communality/Uniqueness extraction →
        Thomson MMSE score projection → Diagnostic telemetry.

    Parameters
    ----------
    df : pd.DataFrame
        Empirical observation data frame (arbitrary numeric + non-numeric columns).
    k : int
        Target number of latent factors to extract.
    zero_var_floor : float
        Minimum standard deviation boundary for zero-variance channels.

    Returns
    -------
    FAResult
        Fully populated result container.

    Raises
    ------
    ValueError
        If dimensionality constraints are violated (n ≤ m or k ≥ m).
    """
    # Layer 1 — Z-score standardization
    Z, labels = zscore_standardize(df, zero_var_floor)
    n, m = Z.shape

    # Layer 2 — Dimensionality constraint validation
    validate_dimensions(n, m, k)

    # Layer 3 — SVD decomposition + Varimax rotation
    L_rot, h2, phi2, T = fit_factor_analysis(Z, k)

    # Layer 4 — Correlation matrix (for Thomson regression)
    R = (Z.T @ Z) / (n - 1)

    # Layer 5 — Thomson MMSE latent score projection
    F = project_factor_scores(Z, L_rot, R)

    # Layer 6 — Factor score empirical variance (ddof=1)
    factor_vars = np.var(F, axis=0, ddof=1)

    # Layer 7 — Diagnostic telemetry
    avg_comm_pct, avg_uniq_pct = print_diagnostics(h2, phi2, factor_vars, labels)

    return FAResult(
        loadings_rotated    = L_rot,
        communalities       = h2,
        uniquenesses        = phi2,
        factor_scores       = F,
        factor_score_vars   = factor_vars,
        sensor_labels       = labels,
        n_obs               = n,
        n_features          = m,
        n_factors           = k,
        avg_communality_pct = avg_comm_pct,
        avg_uniqueness_pct  = avg_uniq_pct,
        rotation_matrix     = T,
    )


# ─────────────────────────────────────────────────────────────────────────────
# 9. 2 × 2 PLOTLY DIAGNOSTIC DASHBOARD
# ─────────────────────────────────────────────────────────────────────────────

def build_diagnostic_dashboard(result: FAResult) -> go.Figure:
    """
    Render a 2×2 asymmetric diagnostic canvas using Plotly subplots.

    Grid layout:
        (1,1) Structural Loadings Heatmap     |  (1,2) Variance Partitioning Bars
        (2,1) Sensor Noise Floor Line          |  (2,2) Latent Factor Variance Bars

    Parameters
    ----------
    result : FAResult
        Populated FA result object.

    Returns
    -------
    fig : go.Figure
        Fully configured Plotly figure.
    """
    r = result
    labels  = r.sensor_labels
    k       = r.n_factors
    abs_L   = np.abs(r.loadings_rotated)   # (m, k)
    comm_pct = r.communalities * 100
    uniq_pct = r.uniquenesses  * 100
    factor_labels = [f"Factor {i+1}" for i in range(k)]

    # ── Figure scaffold ──────────────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=2,
        horizontal_spacing=0.24,
        vertical_spacing=0.28,
        subplot_titles=[
            "Structural Loadings |λ|",
            "Variance Partitioning Profile",
            "Sensor Noise Floor φ²",
            "Latent Factor Variance",
        ],
    )

    # ── (1,1) Heatmap — |Λ_rotated| with YlOrRd scale ──────────────────────
    heatmap = go.Heatmap(
        z=abs_L,
        x=factor_labels,
        y=labels,
        colorscale="YlOrRd",
        zmin=0.0,
        zmax=1.0,
        text=np.round(abs_L, 2),
        texttemplate="%{text}",
        textfont={"size": 9},
        showscale=True,
        colorbar=dict(
            x=-0.15,
            y=0.78,
            yanchor="middle",
            xanchor="right",
            len=0.38,
            title=dict(text="|λ|", side="right"),
            thickness=12,
            tickfont=dict(size=9),
        ),
        name="Loadings",
    )
    fig.add_trace(heatmap, row=1, col=1)

    # ── (1,2) Stacked horizontal bar — communality + uniqueness ─────────────
    fig.add_trace(
        go.Bar(
            y=labels,
            x=comm_pct,
            orientation="h",
            name="Communality h²",
            marker_color="#1f77b4",
            showlegend=True,
        ),
        row=1, col=2,
    )
    fig.add_trace(
        go.Bar(
            y=labels,
            x=uniq_pct,
            orientation="h",
            name="Uniqueness φ²",
            marker_color="#ff7f0e",
            showlegend=True,
        ),
        row=1, col=2,
    )
    fig.update_xaxes(range=[0, 100], row=1, col=2)
    fig.update_layout(barmode="stack")

    # ── (2,1) Noise floor line — φ² dot-dashed red with × markers ───────────
    fig.add_trace(
        go.Scatter(
            x=labels,
            y=r.uniquenesses,
            mode="lines+markers",
            name="Noise Floor φ²",
            line=dict(color="#d62728", width=2, dash="dashdot"),
            marker=dict(symbol="x", size=9, color="#d62728", line=dict(width=2)),
            showlegend=True,
        ),
        row=2, col=1,
    )
    fig.update_xaxes(tickangle=25, row=2, col=1)
    fig.update_yaxes(range=[0, 1], row=2, col=1)

    # ── (2,2) Factor variance vertical bars — green with black border ────────
    fig.add_trace(
        go.Bar(
            x=factor_labels,
            y=r.factor_score_vars,
            name="Factor Variance",
            marker=dict(
                color="#2ca02c",
                line=dict(color="black", width=0.5),
            ),
            showlegend=True,
        ),
        row=2, col=2,
    )

    # ── Global layout ─────────────────────────────────────────────────────────
    fig.update_layout(
        width=1250,
        height=750,
        template="plotly_white",
        title=dict(
            text=(
                f"Factor Analysis Diagnostic Dashboard  —  "
                f"k={k} factors  |  "
                f"Avg Communality={r.avg_communality_pct:.1f}%  |  "
                f"Avg Uniqueness={r.avg_uniqueness_pct:.1f}%"
            ),
            font=dict(size=13),
            x=0.5,
        ),
        legend=dict(
            orientation="h",
            x=0.5,
            y=1.02,
            xanchor="center",
            yanchor="bottom",
            font=dict(size=11),
        ),
        margin=dict(t=150, b=60, l=140, r=80),
        font=dict(family="Arial, sans-serif", size=11),
    )

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# 10. SYNTHETIC DATA GENERATOR (for standalone demo)
# ─────────────────────────────────────────────────────────────────────────────

def generate_synthetic_data(
    n: int = 200,
    sensor_config: Optional[dict] = None,
    noise_scale: float = 0.30,
    random_seed: int = 42,
) -> pd.DataFrame:
    """
    Generate a synthetic multi-sensor observation data frame.

    Columns are constructed as linear mixtures of k latent factors
    plus independent Gaussian noise, producing realistic factor
    structure for pipeline validation.

    Parameters
    ----------
    n : int
        Number of observation snapshots.
    sensor_config : dict, optional
        Maps sensor group names to factor loading vectors.
        Default: industrial 8-channel configuration.
    noise_scale : float
        Standard deviation of additive Gaussian noise.
    random_seed : int
        NumPy random seed for reproducibility.

    Returns
    -------
    pd.DataFrame
        Synthetic sensor observation data frame of shape (n, m).
    """
    rng = np.random.default_rng(random_seed)

    if sensor_config is None:
        sensor_config = {
            # channel_name: [factor1_loading, factor2_loading, factor3_loading]
            "Temp_A":   [0.85, 0.08, 0.07],
            "Temp_B":   [0.82, 0.10, 0.08],
            "Press_1":  [0.09, 0.88, 0.10],
            "Press_2":  [0.07, 0.79, 0.09],
            "Vib_X":    [0.06, 0.12, 0.84],
            "Vib_Y":    [0.08, 0.09, 0.81],
            "Flow_M":   [0.11, 0.11, 0.77],
            "RPM":      [0.07, 0.08, 0.72],
        }

    channel_names = list(sensor_config.keys())
    loadings_mat  = np.array(list(sensor_config.values()))  # (m, k)
    k = loadings_mat.shape[1]

    # Latent factor realizations
    F_true = rng.standard_normal((n, k))

    # Observation matrix: X = F_true @ Λᵀ + noise
    X = F_true @ loadings_mat.T + noise_scale * rng.standard_normal((n, len(channel_names)))

    return pd.DataFrame(X, columns=channel_names)


# ─────────────────────────────────────────────────────────────────────────────
# 11. ENTRYPOINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # ── Generate synthetic industrial sensor data ────────────────────────────
    log.info("=" * 60)
    log.info("  PARAMETRIC FACTOR PARTITION ENGINE  —  Demo Run")
    log.info("=" * 60)

    df = generate_synthetic_data(n=200, noise_scale=0.30, random_seed=42)
    log.info("[DATA]     Shape: %s  |  Channels: %s", df.shape, df.columns.tolist())

    # ── Run the FA engine ────────────────────────────────────────────────────
    result = run_factor_analysis(df, k=3)

    # ── Build and display the 2×2 diagnostic dashboard ──────────────────────
    fig = build_diagnostic_dashboard(result)
    fig.show()

    # ── Optional: export to HTML / PNG ──────────────────────────────────────
    # fig.write_html("fa_diagnostic_dashboard.html")
    # fig.write_image("fa_diagnostic_dashboard.png", scale=2)